In [9]:
import os
import json
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

from xgboost import XGBClassifier

DATA_PATH = "../data/processed_loans.csv"
os.makedirs("../models", exist_ok=True)

df = pd.read_csv(DATA_PATH)
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X.columns.tolist()


['income_annum',
 'emi_income_ratio',
 'no_of_dependents',
 'is_not_graduate',
 'is_self_employed',
 'asset_total',
 'cibil_band']

In [10]:
lr = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)
lr_proba = lr.predict_proba(X_test)[:, 1]

print("Logistic Regression Accuracy:", round(accuracy_score(y_test, lr_pred), 4))
print("Logistic Regression ROC AUC:", round(roc_auc_score(y_test, lr_proba), 4))
print(classification_report(y_test, lr_pred))


Logistic Regression Accuracy: 0.9836
Logistic Regression ROC AUC: 0.9944
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       323
           1       0.99      0.98      0.99       531

    accuracy                           0.98       854
   macro avg       0.98      0.98      0.98       854
weighted avg       0.98      0.98      0.98       854



In [11]:
feature_names = X.columns.tolist()

constraints_map = {
    "income_annum": 1,
    "emi_income_ratio": -1,
    "no_of_dependents": -1,
    "is_not_graduate": -1,
    "is_self_employed": 1,
    "asset_total": 1,
    "cibil_band": 1,
}
monotone_constraints = tuple(constraints_map[f] for f in feature_names)

xgb = XGBClassifier(
    n_estimators=600,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=2.0,
    reg_alpha=0.5,
    min_child_weight=3,
    objective="binary:logistic",
    eval_metric="logloss",
    monotone_constraints=monotone_constraints,
    random_state=42,
)

xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)[:, 1]

print("XGBoost Accuracy:", round(accuracy_score(y_test, xgb_pred), 4))
print("XGBoost ROC AUC:", round(roc_auc_score(y_test, xgb_proba), 4))
print(classification_report(y_test, xgb_pred))


XGBoost Accuracy: 0.966
XGBoost ROC AUC: 0.9707
              precision    recall  f1-score   support

           0       0.92      0.99      0.96       323
           1       0.99      0.95      0.97       531

    accuracy                           0.97       854
   macro avg       0.96      0.97      0.96       854
weighted avg       0.97      0.97      0.97       854



In [12]:
joblib.dump(xgb, "../models/xgb_model.pkl")

with open("../models/feature_columns.json", "w") as f:
    json.dump(feature_names, f, indent=2)

print("Saved models/xgb_model.pkl and models/feature_columns.json")


Saved models/xgb_model.pkl and models/feature_columns.json
